In [2]:
from IPython import display, HTML

# Define a largura desejada em pixels
largura = 1000

# Configura a largura da área de exibição
display(HTML(f"<style>.container {{ width:{largura}px !important; }}</style>"))

ImportError: cannot import name 'HTML' from 'IPython' (C:\ProgramData\anaconda3\Lib\site-packages\IPython\__init__.py)

In [1]:
from IPython.display import display, HTML
display(HTML("<style>.container { width:150% !important; }</style>"))
import warnings
warnings.simplefilter('ignore')

import pandas as pd
import matplotlib.pyplot as plot
import matplotlib.dates as mdates
import seaborn as sns
from plotly.offline import plot as plt
import plotly.graph_objs as go
import plotly.io as pio
import time
from datetime import datetime, timedelta, date
import teradata
import  businessdays as bzd

In [ ]:

start = time.time() 
system = "TERADBAWS.sicredi.net"
user = "P_USR_DL_GESTAO_RECURSOS"
passw = "JAW0b71329aadade35365afc6X"
udaExec = teradata.UdaExec (appName="TESTE", version="1.0", logConsole=False)
session = udaExec.connect(method="odbc", system=system,
        username=user, password=passw, Authentication="TDNEGO");


select_feriados = "Select DT from dl_Gestao_recursos.bas_cad_fer where tipo in ('3','2')"
feriados = pd.read_sql(select_feriados, session)
cal = bzd.Calendar(holidays=feriados.DT,weekdays=['Sunday','Saturday'])

data_inicial = cal.offset(datetime.now(),-180  )###DATA PADRÃO D-180
data_inicial = data_inicial.strftime('%Y-%m-%d') ###DATA PADRÃO D-180

data_final = cal.offset(datetime.now(),-1  )###DATA PADRÃO D-1
data_final = data_final.strftime('%Y-%m-%d') ###DATA PADRÃO D-1

data_a = cal.offset(data,-5).strftime('%Y-%m-%d') ###DATA PADRÃO D-1
data_b = cal.offset(data,-20).strftime('%Y-%m-%d') ###DATA PADRÃO D-1
data_c = cal.offset(data,-60).strftime('%Y-%m-%d') ###DATA PADRÃO D-1
data_d = cal.offset(data,-50).strftime('%Y-%m-%d') ###DATA PADRÃO D-1


#data_inicial='2023-01-01'
#data_final='2023-07-25'

In [ ]:
Select = f"""
        SELECT VAR.DTPOS, VAR.NOMECARTEIRA, VAR.CODCARTEIRAN2, VAR.CODCARTEIRAN3, DECODE(VAR.GRUPO,'Total', 'TOTAL',DECODE(VAR.CODCARTEIRAN2,'Arbitragem', 'ARBITRAGEM',VAR.CODCARTEIRAN3)) as Estrategia, VAR.var, VAR.var21, VAR.estresse, VAR.Cvar, VAR.ES 
        ,PL.VL_PATRLIQTOT1, VAR.CVAR/PL.VL_PATRLIQTOT1 AS CVAR_PL, VAR/PL.VL_PATRLIQTOT1 AS VAR_PL
        FROM DL_GESTAO_RECURSOS.VAR_ESTRATEGIA_SAS VAR
        left join dl_Gestao_recursos.sac_cl_patrsint pl on pl.dt = VAR.DTPOS AND VAR.NOMECARTEIRA = PL.CLCLI_CD
        WHERE VAR.DTPOS >= '2024-11-01' AND VAR.NOMECARTEIRA = 'FM MACRO ALOC' and ((VAR.codproduto is null and (case when VAR.CODCARTEIRAN2 = 'Arbitragem' then VAR.CODCARTEIRAN3 ELSE NULL END IS NULL)
        and DECODE(VAR.CODCARTEIRAN2,'Arbitragem', VAR.CODCARTEIRAN2,VAR.CODCARTEIRAN3) is not null) OR (VAR.GRUPO = 'Total'))
        """

df_var = pd.read_sql(Select, session) 